In [ ]:
import numpy as np
import pickle
import matplotlib.pyplot as plt
import matplotlib.colors as colors
import json
from sklearn.decomposition import PCA

In [ ]:
DISEASE = input("Disease: ")
OUTPUT_DIRECTORY = f"output/{DISEASE}/"
NONE_OUTPUT_DIRECTORY = f"output/NONE/"
DGIDB_DIRECTORY = f"../Gen_Hypergraph/output/DGIDB_{DISEASE}/"
MSIGDB_DIRECTORY = "../Gen_Hypergraph/output/MSigDB_Full/"

In [ ]:
# load selected communities
with open(f"{OUTPUT_DIRECTORY}/result_communities_selected.pkl", "rb") as f:
    communities_selected = pickle.load(f)

In [ ]:
with open(DGIDB_DIRECTORY + "gene_to_index.json", "r") as file:
    dgidb = json.load(file)
with open(MSIGDB_DIRECTORY + "gene_to_index.json", "r") as file:
    msigdb = json.load(file)
with open(OUTPUT_DIRECTORY + "gene_to_index_distinct.json", "r") as file:
    gene_to_index_distinct = json.load(file)

In [ ]:
index_to_gene_distinct = {index: gene for gene, index in gene_to_index_distinct.items()}

In [ ]:
# concatenate all genes from selected communities
all_genes = []
for community in communities_selected:
    all_genes.extend(community)
print(all_genes)

In [ ]:
len(all_genes)

In [ ]:
all_genes_M = [msigdb.get(index_to_gene_distinct[gene_idx]) for gene_idx in all_genes]

In [ ]:
# DGIDB-only genes are not in selected communities
len(all_genes_M)

In [ ]:
# with open(DGIDB_DIRECTORY + "gene_to_index.json", "r") as file:
#     dgidb = json.load(file)

In [ ]:
# all_genes = list(range(len(dgidb)))

In [ ]:
# load eigenvalues and eigenvectors
eigenvalues = np.load(OUTPUT_DIRECTORY + "/eigenvalues/vals_2.npy")
eigenvectors = np.load(OUTPUT_DIRECTORY + "/eigenvectors/vecs_2.npy")

In [ ]:
# # load eigenvalues and eigenvectors single-layer
# eigenvalues_M = np.load(OUTPUT_DIRECTORY + "eigenvalues_M.npy")
# eigenvectors_P_M = np.load(OUTPUT_DIRECTORY + "eigenvectors_P_M.npy")

In [ ]:
# load eigenvalues and eigenvectors single-layer
eigenvalues_M = np.load(NONE_OUTPUT_DIRECTORY + "/eigenvalues/vals_2.npy")
eigenvectors_M = np.load(NONE_OUTPUT_DIRECTORY + "/eigenvectors/vecs_2.npy")

In [ ]:
# setting universal font sizes
font_size = 20
tick_font_size = 16
plt.rcParams.update({
    "font.size": font_size,          # base font size
    "axes.titlesize": font_size,
    "axes.labelsize": font_size,
    "xtick.labelsize": tick_font_size,
    "ytick.labelsize": tick_font_size,
    "legend.fontsize": font_size,
    "figure.titlesize": font_size,
    "legend.loc": 'best'
})


# PCA

In [ ]:
# load ddm
ddm_both = np.load(f"{OUTPUT_DIRECTORY}/ddm_[2, 4, 6, 8].npy")

In [ ]:
# load npy arrays for NONE
ddm_msigdb = np.load(f"{NONE_OUTPUT_DIRECTORY}/ddm_[2, 4, 6, 8].npy")

In [ ]:
# ddm_both: diffusion distance matrix using both sources
# ddm_msigdb: diffusion distance matrix using MSigDB only

pca_both = PCA(n_components=2)
Phi_pca = pca_both.fit_transform(ddm_both)

pca_msigdb = PCA(n_components=2)
Phi_pca_M = pca_msigdb.fit_transform(ddm_msigdb)

print("Explained variance ratio, both:", pca_both.explained_variance_ratio_)
print("Explained variance ratio, MSigDB:", pca_msigdb.explained_variance_ratio_)

# === MODIFICATION START: remove outliers for visualization using percentile cutoff ===
pct = 99  # changed: keep points inside the 1st--99th percentile range

x1, y1 = Phi_pca[:, 0], Phi_pca[:, 1]
x2, y2 = Phi_pca_M[:, 0], Phi_pca_M[:, 1]

lo_x1, hi_x1 = np.percentile(x1, [100 - pct, pct])
lo_y1, hi_y1 = np.percentile(y1, [100 - pct, pct])

lo_x2, hi_x2 = np.percentile(x2, [100 - pct, pct])
lo_y2, hi_y2 = np.percentile(y2, [100 - pct, pct])

keep_both = (
    (x1 >= lo_x1) & (x1 <= hi_x1) &
    (y1 >= lo_y1) & (y1 <= hi_y1)
)

keep_msigdb = (
    (x2 >= lo_x2) & (x2 <= hi_x2) &
    (y2 >= lo_y2) & (y2 <= hi_y2)
)
# === MODIFICATION END ===

plt.figure(figsize=(7, 5))

# === MODIFICATION START: plot filtered PCA coordinates with labels ===
plt.scatter(
    Phi_pca[keep_both, 0],
    Phi_pca[keep_both, 1],
    s=8,
    alpha=0.8,
    label="Both"
)

plt.scatter(
    Phi_pca_M[keep_msigdb, 0],
    Phi_pca_M[keep_msigdb, 1],
    s=8,
    alpha=0.8,
    label="MSigDB"
)
# === MODIFICATION END ===

plt.xlabel("PC1")
plt.ylabel("PC2")
plt.legend(loc="upper right", fontsize=15, markerscale=2, frameon=True)
plt.tight_layout()
plt.savefig(f"../Graphs/{DISEASE}/PCA_ddm_comparison.png")
plt.show()

# Top eigenvalues magnitude

In [ ]:
top_eigenvalues = eigenvalues[:20]
top_eigenvalues_M = eigenvalues_M[:20]

In [ ]:
font_size = 25
tick_font_size = 16
plt.rcParams.update({
    "font.size": font_size,          # base font size
    "axes.titlesize": font_size,
    "axes.labelsize": font_size,
    "xtick.labelsize": tick_font_size,
    "ytick.labelsize": tick_font_size,
    "legend.fontsize": font_size,
    "figure.titlesize": font_size,
    "legend.loc": 'best'
})


In [ ]:
# plot eigenvalues comparison, dot plot, blue for multilayer, orange for single-layer
plt.figure(figsize=(10,6))
plt.scatter(range(len(top_eigenvalues)), top_eigenvalues, label="Multilayer", color='blue')
plt.scatter(range(len(top_eigenvalues_M)), top_eigenvalues_M, label="Single-layer", color='orange')
plt.xlabel("Index")
plt.ylabel("Eigenvalue")
plt.title("Top 20 Eigenvalues Comparison")
# put legend outside the plot
plt.legend()
plt.savefig(f"../Graphs/{DISEASE}/top_20_eigenvalues_comparison.png")

# 2D eigenvalues embedding plots

In [ ]:
lambda1,lambda2 = eigenvalues[1],eigenvalues[2]
phi1,phi2 = eigenvectors[:,1],eigenvectors[:,2]

In [ ]:
lambda1_M,lambda2_M = eigenvalues_M[1],eigenvalues_M[2]
phi1_M,phi2_M = eigenvectors_M[:,1],eigenvectors_M[:,2]

In [ ]:
def embedding(gene_idx,t):
    return np.array([(lambda1**t)*phi1[gene_idx], 
                     (lambda2**t)*phi2[gene_idx]])

In [ ]:
def embedding_M(gene_idx,t):
    return np.array([(lambda1_M**t)*phi1_M[gene_idx], 
                     (lambda2_M**t)*phi2_M[gene_idx]])

In [ ]:
embeddings = np.array([embedding(gene_idx, t=6) for gene_idx in all_genes])
print(embeddings)

In [ ]:
embeddings_M = np.array([embedding_M(gene_idx, t=6) for gene_idx in all_genes_M])
print(embeddings_M)

In [ ]:
x,y=embeddings[:,0],embeddings[:,1]

In [ ]:
x_M,y_M=embeddings_M[:,0],embeddings_M[:,1]

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle, ConnectionPatch

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 7))

# ----- full plot on the left -----
ax1.scatter(x, y, label="Scatter 1")  # unchanged
ax1.scatter(x_M, y_M, label="Scatter 2")  # unchanged
ax1.set_xlabel("Eigenvalue 1")
ax1.set_ylabel("Eigenvalue 2")
ax1.set_title("Full view")  # unchanged
ax1.legend(fontsize=15)  # changed: create the legend only once, after plotting, with smaller font

# ----- choose zoom region -----
x_min, x_max = min(x_M)-0.0005, max(x_M)+0.0005  # unchanged
y_min, y_max = min(y_M)-0.0005, max(y_M)+0.0005  # unchanged

# draw rectangle on left plot
rect = Rectangle((x_min, y_min), x_max - x_min, y_max - y_min,
                 fill=False, edgecolor='red', linewidth=2)  # unchanged
ax1.add_patch(rect)  # unchanged

# ----- zoomed plot on the right -----
ax2.scatter(x, y, label="Scatter 1")  # unchanged
ax2.scatter(x_M, y_M, label="Scatter 2")  # unchanged
ax2.set_xlim(x_min, x_max)  # unchanged
ax2.set_ylim(y_min, y_max)  # unchanged
ax2.set_title("Zoomed view")  # unchanged
ax2.legend(fontsize=15)  # changed: actually create the right legend here, after plotting

# ----- connector lines between box and zoomed panel -----
con1 = ConnectionPatch(xyA=(x_max, y_max), coordsA=ax1.transData,
                       xyB=(x_min, y_max), coordsB=ax2.transData,
                       color="red", linewidth=1.5)  # unchanged
con2 = ConnectionPatch(xyA=(x_max, y_min), coordsA=ax1.transData,
                       xyB=(x_min, y_min), coordsB=ax2.transData,
                       color="red", linewidth=1.5)  # unchanged
fig.add_artist(con1)  # unchanged
fig.add_artist(con2)  # unchanged

plt.tight_layout()
plt.savefig(f"../Graphs/{DISEASE}/eigen_embedding_comparison.png")
plt.show()

In [ ]:
plt.figure(figsize=(10,8))
plt.scatter(x_M, y_M, color = '#ff7f0e', label="Single-layer")
plt.xlabel("Eigenvalue 1")
plt.ylabel("Eigenvalue 2")
plt.savefig(f"../Graphs/{DISEASE}/eigen_embedding_single_layer.png")

In [ ]:
plt.figure(figsize=(10,8))
plt.scatter(x, y, color = '#1f77b4', label="Single-layer")
plt.xlabel("Eigenvalue 1")
plt.ylabel("Eigenvalue 2")
plt.savefig(f"../Graphs/{DISEASE}/eigen_embedding_multi_layer.png")